# Exp96 Method C: Kaggle T4 engineering smoke

**Engineering evidence only.** This notebook never submits official parents or Slurm jobs. It reuses the repository's four development smoke commands: 20,000 agent interactions, full ten-target/2,000-step probes at 10k and 20k, and complete checkpoints. The T4 may be unable to run the A100 three-process pattern. An `INCOMPLETE` report identifies what was not tested. It cannot satisfy the source-bound A100 smoke or scientific-calibration approval gates.

Before running: create a Kaggle notebook, select **Accelerator → GPU T4**, turn **Internet on**, and choose **Save Version → Save & Run All** for an unattended committed run. Outputs are written under `/kaggle/working/`. Kaggle may impose session quotas; the browser can close during a committed run, but inspect the saved version's final status and output rather than assuming it finished.


In [ ]:
import json, os, re, subprocess, sys
from pathlib import Path
WORK = Path('/kaggle/working')
assert WORK.is_dir(), 'This notebook must run in Kaggle'
smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and 'T4' in smi.stdout, 'Select a Kaggle NVIDIA T4 accelerator'
print('Notebook kernel Python:', sys.version)


In [ ]:
REPO_URL = 'https://github.com/kaves29/scaling-drl-research.git'
BRANCH = 'codex/exp96-kaggle-t4-smoke'
BASE_SHA = 'bd568d482c261f067add0b1d44bb57a6aceb52b1'
remote = subprocess.check_output(['git', 'ls-remote', REPO_URL, f'refs/heads/{BRANCH}'], text=True).split()
assert len(remote) == 2 and re.fullmatch(r'[0-9a-f]{40}', remote[0]), 'Published notebook branch unavailable'
SOURCE_SHA = remote[0]
REPO = WORK / f'scaling-drl-research_{SOURCE_SHA}'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', SOURCE_SHA], check=True)
subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor', BASE_SHA, SOURCE_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_SHA
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=all'], text=True).strip()
MAIN = REPO / 'main'
for required in ('run.py', 'configs/base_exp12_ten.yaml', 'scripts/exp96_gpu_smoke.py', 'scripts/exp96_kaggle_t4_smoke.py', 'requirements.txt'):
    assert (MAIN / required).is_file(), required
print('Exact clean source SHA:', SOURCE_SHA)
print('Experiment directory:', MAIN)


In [ ]:
# Install into a separate Python 3.12.13 environment; leave Kaggle's kernel intact.
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
UV = shutil.which('uv') or str(Path(sys.executable).parent / 'uv')
assert Path(UV).is_file(), 'uv CLI missing after installation'
subprocess.run([UV, 'python', 'install', '3.12.13'], check=True)
VENV = WORK / 'exp96_py31213'
subprocess.run([UV, 'venv', '--python', '3.12.13', str(VENV)], check=True)
PY = VENV / 'bin/python'
subprocess.run([UV, 'pip', 'install', '--python', str(PY), '-r', str(MAIN / 'requirements.txt'), 'jax[cuda12]==0.4.34'], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(PY)], check=True)
print('Pinned runtime:', subprocess.check_output([str(PY), '--version'], text=True).strip())


In [ ]:
# This invokes a real JAX JIT matmul and asserts that its output remains GPU-backed.
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0', JAX_PLATFORMS='cuda,cpu', JAX_ENABLE_X64='false', MUJOCO_GL='disable',
           XLA_PYTHON_CLIENT_PREALLOCATE='false', PYTHONDONTWRITEBYTECODE='1',
           PYTHONUNBUFFERED='1', WANDB_MODE='offline')
for key in ('JAX_PLATFORM_NAME', 'NVIDIA_TF32_OVERRIDE', 'XLA_FLAGS', 'XLA_PYTHON_CLIENT_MEM_FRACTION'):
    ENV.pop(key, None)
preflight = subprocess.run([str(PY), '-c',
    'import json; from scripts.exp96_kaggle_t4_smoke import gpu_preflight; print(json.dumps(gpu_preflight(), indent=2))'],
    cwd=MAIN, env=ENV, text=True, capture_output=True)
print(preflight.stdout)
print(preflight.stderr)
assert preflight.returncode == 0, 'CUDA, precision, pinned dependencies, or environment preflight failed'


In [ ]:
# Bounded five-hour engineering run. A nonzero exit is retained as FAIL/INCOMPLETE evidence.
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = WORK / f'exp96_t4_{SOURCE_SHA}_{stamp}'
CONSOLE = WORK / f'exp96_t4_{SOURCE_SHA}_{stamp}_console.log'
cmd = [str(PY), '-u', '-m', 'scripts.exp96_kaggle_t4_smoke', '--root', str(RUN_ROOT),
       '--commit', SOURCE_SHA, '--max-seconds', '18000', '--phase-seconds', '2700']
print('Running:', ' '.join(cmd), flush=True)
with CONSOLE.open('w') as saved:
    proc = subprocess.Popen(cmd, cwd=MAIN, env=ENV, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
        saved.write(line)
        saved.flush()
    EXIT_CODE = proc.wait()
print('Runner exit:', EXIT_CODE, '(0=PASS, 3=INCOMPLETE, 2=FAIL)')
assert (RUN_ROOT / 'summary.json').is_file(), 'Runner ended without a structured receipt; inspect console log'


In [ ]:
from IPython.display import display, FileLink
summary = json.loads((RUN_ROOT / 'summary.json').read_text())
print(json.dumps({key: summary.get(key) for key in
                  ('source_commit', 'verdict', 'coverage', 'limits', 'error', 'elapsed_seconds')}, indent=2))
assert summary['source_commit'] == SOURCE_SHA
bundle = WORK / f'{RUN_ROOT.name}_evidence.tar.gz'
assert bundle.is_file(), 'Evidence bundle missing'
print('Upload this bundle for independent review:', bundle)
display(FileLink(str(bundle)))
display(FileLink(str(bundle) + '.sha256'))
display(FileLink(str(CONSOLE)))


## Interpreting the result

`PASS` means the full development smoke pattern completed on this T4. `INCOMPLETE` means hardware, time, or missing cases prevented full coverage; inspect per-case receipts and `limits`. `FAIL` means setup, backend, source, or a completed scientific artifact check failed. The archive includes actual attempt logs, source/configuration copies, package versions, resource samples, probe NPZ/CSV files, run metadata, local results, W&B offline output where generated, and checkpoint metadata/inventory. Full Orbax checkpoint tensors remain in `/kaggle/working/` and are intentionally omitted from the compact bundle; retain the notebook output version if they may be needed later.

Download `*_evidence.tar.gz` from the final cell or the Kaggle notebook's **Output** tab and attach it to the research conversation. Include the console log if the notebook stopped before producing a bundle. This test never certifies Delta A100 packing, scientific calibration, or official parent eligibility.
